# Sprint 4: RAG Prompt, Answer, and Evaluation

This notebook demonstrates the full Sprint 4 backend flow for the Mini Wikipedia RAG system.

## What this notebook covers
1. Confirm the API server is running
2. Initialize the vector index
3. Construct an augmented prompt from retrieved documents
4. Generate an answer with GPT-4o
5. Evaluate answer quality against reference answers

In [1]:
import sys
sys.path.insert(0, '../src')

import json
import requests

BASE_URL = 'http://127.0.0.1:8000'
QUERY_TEXT = 'What is Python?'
TOP_K = 3

print(f'Using server at {BASE_URL}')
print(f'Test query: {QUERY_TEXT}')

Using server at http://127.0.0.1:8000
Test query: What is Python?


## 1. Check Server Readiness

Confirm that the FastAPI server is reachable and has been restarted with the Sprint 4 routes.

In [2]:
try:
    response = requests.get(f'{BASE_URL}/openapi.json')
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        routes = response.json().get('paths', {})
        print('\nSprint 4 routes present:')
        for route in ['/rag/prompt', '/rag/answer', '/rag/evaluate']:
            print(f'  {route}: {route in routes}')
    else:
        print(f'Error {response.status_code}: {response.text[:200]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
    print('   Start with: uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000 --reload')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200

Sprint 4 routes present:
  /rag/prompt: True
  /rag/answer: True
  /rag/evaluate: True


## 2. Initialize the Vector Index

Load the sample documents and create the vector index used by the RAG endpoints.

In [3]:
try:
    response = requests.post(f'{BASE_URL}/ingest', params={'document_count': 10})
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        ingest_result = response.json()
        print(json.dumps(ingest_result, indent=2))
        print(f'\n✓ Ingested {ingest_result["documents_ingested"]} documents')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')
        print('\n⚠️ If ingestion fails here, check Azure auth: azd auth login --scope api://ailab/Model.Access')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
{
  "status": "success",
  "documents_ingested": 10,
  "index_ready": true
}

✓ Ingested 10 documents


## 3. Construct the Augmented Prompt

Inspect the prompt that combines the user query with the retrieved documents.

In [4]:
prompt_payload = {'query': QUERY_TEXT, 'top_k': TOP_K}
prompt_result = None

try:
    response = requests.post(f'{BASE_URL}/rag/prompt', json=prompt_payload)
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        prompt_result = response.json()
        print(f'Embedding dimension: {prompt_result["query_embedding_dimension"]}')
        print(f'Retrieved documents: {len(prompt_result["documents"])}')
        print('\nPrompt preview:\n')
        print(prompt_result['prompt'][:1200])
    elif response.status_code == 404:
        print('❌ The Sprint 4 endpoints are not available on the running server.')
        print('   Restart with: uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000 --reload')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
Embedding dimension: 3072
Retrieved documents: 3

Prompt preview:

You are a retrieval-augmented assistant. Answer the user question using only the provided context. If the context is insufficient, say that directly.

User Question:
What is Python?

Retrieved Context:
[1] Sample Document 0 score=0.534
Python is a high-level, interpreted programming language created by Guido van Rossum. First released in 1991, Python's design philosophy emphasizes code readability.

[2] Sample Document 7 score=0.160
Natural language processing is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language.

[3] Sample Document 3 score=0.158
Data science is an interdisciplinary field that uses scientific methods, processes, algorithms and systems to extract knowledge and insights from structured and unstructured data.

Write a concise answer grounded in the retrieved context.


## 4. Generate an Answer

Call GPT-4o through the Sprint 4 answer endpoint and inspect the grounded answer returned by the backend.

In [5]:
answer_payload = {'query': QUERY_TEXT, 'top_k': TOP_K}
answer_result = None

try:
    response = requests.post(f'{BASE_URL}/rag/answer', json=answer_payload)
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        answer_result = response.json()
        print('Generated answer:\n')
        print(answer_result['answer'])
        print(f'\nRetrieved documents: {len(answer_result["documents"])}')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
Generated answer:

Python is a high-level, interpreted programming language created by Guido van Rossum, first released in 1991. Its design philosophy emphasizes code readability.

Retrieved documents: 3


## 5. Evaluate the RAG Pipeline

Run the lightweight evaluation endpoint to compare generated answers against reference answers.

In [6]:
try:
    response = requests.get(f'{BASE_URL}/rag/evaluate', params={'limit': 3, 'top_k': TOP_K})
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        evaluation_result = response.json()
        print(json.dumps(evaluation_result, indent=2))
        print(f'\nAverage score: {evaluation_result["average_score"]}')
        print(f'Passed examples: {evaluation_result["passed_count"]}/{evaluation_result["example_count"]}')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
{
  "example_count": 3,
  "average_score": 0.955,
  "passed_count": 3,
  "results": [
    {
      "query": "What is Python?",
      "expected_answer": "Python is a high-level, interpreted programming language created by Guido van Rossum.",
      "generated_answer": "Python is a high-level, interpreted programming language created by Guido van Rossum, first released in 1991. Its design philosophy emphasizes code readability.",
      "score": 1.0
    },
    {
      "query": "What is machine learning?",
      "expected_answer": "Machine learning is a subset of artificial intelligence focused on algorithms that improve performance on tasks.",
      "generated_answer": "Machine learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to improve their performance on tasks.",
      "score": 0.933
    },
    {
      "query": "What is the Internet?",
      "expected_answer": "The Internet is a glo

## Sprint 4 Outcome Checklist

If the cells above succeed, Sprint 4 outcomes are covered:
- `/rag/prompt` constructs an enhanced prompt from retrieved documents
- `/rag/answer` sends the prompt to GPT-4o and returns a response
- `/rag/evaluate` compares generated answers against reference answers
- the backend now exposes the complete retrieval-plus-answer path